# 🏦 Credit Card Fraud Detection — Project Explained Simply

### Basics of AI and Machine Learning — Final Project · RSU, April 2026
**Team:** Girts Ziemanis, Ugis Rutins, Maris Zelcs, Evija Sturmane, Jolanta Stutane

---

## Why this project exists in the real world

Credit card fraud is not a theoretical problem. In 2023 alone, global card fraud losses exceeded **$33 billion USD** — and the figure grows every year as more spending moves online. In the US alone, the Federal Trade Commission received over **416,000 credit card fraud reports** in a single year.

Banks and payment processors face a specific, brutal constraint: they must approve or decline every transaction **in under two seconds**, 24 hours a day, 365 days a year — without any human in the loop. A fraud analyst reviewing each purchase by hand would be far too slow and far too expensive. That's why **machine learning** is the only practical tool for this job at scale.

The consequences of getting it wrong are asymmetric:
- A **missed fraud** → real money is gone. The bank typically reimburses the customer, so it hits the bank's balance sheet directly.
- A **false alarm** → a real customer is briefly inconvenienced. Their card gets unblocked with a phone call. Annoying, but fixable in minutes.

This asymmetry — where one mistake is catastrophic and the other is merely inconvenient — is the engine that drives every technical decision in our project.

---

## What we built — in one sentence

> We trained a machine learning model on 340,000 real card transactions, evaluated five different algorithmic approaches, selected the best-performing model, and **deployed it as a live, browser-accessible web application** running in Docker with a REST API backend.

---

## The project in numbers

| Stat | Value |
|------|-------|
| Transactions analysed | **339,607** |
| Fraud cases in dataset | **1,782 (0.52%)** |
| Original features | **15** |
| Engineered features | **39** |
| Models trained & compared | **5** |
| Test set size | **67,922 transactions** |
| Best model fraud catch rate | **84% (299 of 356 frauds)** |
| Best model false alarms | **24 out of 67,922** |
| Best model ROC-AUC | **0.998** |
| Fraud savings vs worst model | **~$39,000 on this test set alone** |

---

## The technical stack

| Layer | Technology | What it does |
|-------|-----------|-------------|
| **Data & modelling** | Python, pandas, scikit-learn, LightGBM, XGBoost | Data cleaning, feature engineering, training all 5 models |
| **Imbalance handling** | imbalanced-learn (SMOTE) | Balances the 99.5% / 0.5% class split during training |
| **API server** | Python (Flask) · `api.py` | Receives a transaction, runs the model, returns a verdict |
| **Web dashboard** | Next.js (TypeScript) · `fraud-ui/` | Visual interface for submitting transactions and seeing predictions |
| **Deployment** | Docker + Docker Compose | Starts the entire system with one command: `docker-compose up` |
| **Notebook environment** | Jupyter + kagglehub | Downloads the dataset automatically, runs all analysis end-to-end |

---

## How to use this notebook

This notebook tells the **full story of our project** in plain, everyday language. There is **no code** here — just explanations.

Every section ends with two things:
- **💬 If someone asks** — a short, ready-made sentence you can say out loud
- **📂 Where to find the code** — a pointer to the exact cell in `ML_work_and_analysis.ipynb`

---

## The story in one paragraph

> We got a spreadsheet of 340,000 credit card purchases from the western US. Only 0.52% were fraud — about 1 in 200. We studied the data and discovered four clear patterns: fraud happens at night, involves larger amounts, targets online stores, and occurs far from the cardholder's home. We turned those insights into handmade features, trained five models from simple to advanced, and found that LightGBM catches 84% of all fraud with only 24 false alarms per 68,000 transactions. In money terms, choosing the right model prevents roughly $39,000 more in losses than the simplest approach — on this test set alone.

---

## What this notebook covers

| # | Topic | The question it answers |
|---|-------|----------------------|
| 0 | Machine Learning Theory | What is ML and how do computers actually learn? |
| 1 | Problem & Motivation | What are we doing and why is it difficult? |
| 2 | Dataset Overview | What data did we get, and what do the columns mean? |
| 3 | Exploratory Data Analysis | What does fraud actually look like in the data? |
| 4 | Feature Engineering | How did we make the data smarter? |
| 5 | Data Preparation | How did we get the data ready for the computer? |
| 6 | Evaluation Metrics | How do we judge if the model is any good? |
| 7 | Model Training | What models did we try and how do they work? |
| 8 | Model Results | Which model won and why? |
| 9 | Threshold Optimisation | Can we fine-tune the model's sensitivity? |
| 10 | Business Impact | What does all this mean in real money? |
| 11 | Key Takeaways & Demo | What are the big lessons and how does the live demo work? |
| 12 | Glossary | Every technical term explained in plain English |


---

# 0. 🧠 Machine Learning Theory — From Zero

*Before we dive into our project, let's make sure everyone understands the tools we used. This section explains what machine learning actually is — no maths required.*

---

## What is Artificial Intelligence (AI)?

**Artificial Intelligence** is the broad idea of making computers do things that normally require human intelligence — like recognising faces, understanding speech, or deciding whether a bank transaction looks suspicious.

Think of AI as an umbrella term. Machine Learning is one of the most important tools inside that umbrella.

---

## What is Machine Learning?

Normally, when we write a computer program, we write **explicit rules**:

> *"IF transaction amount > $1000 AND time is between midnight and 5 AM THEN flag as fraud."*

This works for simple problems. But fraud patterns are complex — they change over time, vary by region, and involve combinations of dozens of factors. Writing every rule by hand is impossible.

**Machine Learning (ML) is a different idea:** instead of writing the rules yourself, you give the computer **thousands of examples** with known answers, and let it **figure out the rules on its own**.

> *"Here are 270,000 transactions. Here are the labels — fraud or not fraud. Find the pattern."*

The computer doesn't understand what a credit card is. It doesn't know what money is. It just sees numbers — and it finds the mathematical patterns that best separate 0s from 1s.

---

## The three ingredients of any ML project

Every machine learning project — including ours — needs exactly three things:

### 1. 📦 Data (with labels)
Examples the computer can learn from. Each example must have:
- **Input features** — the information describing the example (amount, time, location...)
- **A label** — the correct answer (fraud = 1, legitimate = 0)

Without labelled data, there is nothing to learn from.

### 2. ⚙️ An algorithm (the learning method)
The mathematical procedure that finds patterns in the data. Different algorithms make different assumptions and use different strategies. We tried 5 of them.

### 3. 📏 A way to measure success
You need to know when the computer has learned well — and when it hasn't. Without a clear measure, you can't improve anything.

---

## How does a model actually "learn"?

Here is the simplest possible explanation:

1. **Start with a guess** — the model knows nothing, so it makes random guesses at first
2. **Check against the real answer** — compare the guess to the known correct label
3. **Measure the error** — how wrong was the guess?
4. **Adjust** — slightly adjust the model's internal numbers to reduce the error
5. **Repeat** — do this for all 270,000 training examples, many times over

After thousands of adjustments, the errors get smaller and smaller. The model has **learned the patterns**.

---

## What is a model, exactly?

A **model** is just a mathematical function. It takes numbers in → and produces a prediction out.

For our project:
- **Input:** 39 numbers describing one transaction (amount, hour, distance, category flags...)
- **Output:** One number between 0 and 1 — the model's confidence that this is fraud

Think of it like a recipe. The recipe takes ingredients (the 39 numbers) and produces a result (the fraud probability). Training is the process of **figuring out the best recipe** from the data.

---

## Supervised vs Unsupervised Learning

There are two main types of machine learning:

| Type | What it means | Example |
|------|--------------|---------|
| **Supervised Learning** | Training data has correct answers (labels). The model learns to predict the label. | Our project — every transaction is labelled "fraud" or "not fraud" |
| **Unsupervised Learning** | No labels. The model finds hidden structure on its own. | Grouping customers by spending behaviour |

Our project is **supervised learning** — we have labels and we want the model to learn to predict them.

---

## Classification vs Regression

Within supervised learning, there are two sub-types:

| Type | What it predicts | Example |
|------|-----------------|---------|
| **Classification** | A category (one of several fixed options) | Fraud or Not Fraud ← *our task* |
| **Regression** | A continuous number | Predict next month's sales revenue |

We are doing **binary classification** — binary because there are exactly 2 categories.

---

## The ML workflow — what we actually did

Every ML project follows the same basic steps. Here is how they map to our project:

```
Step 1: Get data           → Downloaded 340,000 labelled transactions
Step 2: Explore (EDA)      → Found 4 key fraud patterns
Step 3: Engineer features  → Created 39 useful numeric features
Step 4: Prepare data       → Split 80/20, scaled, balanced with SMOTE
Step 5: Train models       → 5 algorithms trained on the 80% training set
Step 6: Evaluate           → Measured Recall, F1, AUC on the held-out 20%
Step 7: Choose winner      → LightGBM: best F1-Score, best balance
Step 8: Deploy             → Flask API + Next.js dashboard + Docker
```

Every section of this notebook corresponds to one of these steps.

---

## Why can't the computer just memorise the answers?

If you showed the computer all 270,000 training transactions 1,000 times, it would eventually **memorise** every single one — and score perfectly on them. But then it would completely fail on new transactions it had never seen before.

This is called **overfitting** — the model memorised the training data instead of learning the real underlying patterns. It is like a student who memorises every question from last year's exam, then fails when the questions change slightly.

We prevent overfitting by:
- **Keeping a separate test set** the model never sees during training
- **Limiting model complexity** (e.g., limiting how deep a decision tree can grow)
- **Cross-validation** — testing on multiple different held-out slices during training

---

## Probability vs Binary Decision

Our models don't output "fraud" or "not fraud" directly. They output a **probability** — a number between 0 and 1:

- 0.02 = "I'm 2% sure this is fraud" → very likely legitimate
- 0.97 = "I'm 97% sure this is fraud" → very likely fraud
- 0.50 = "I'm completely unsure"

We then apply a **threshold** to convert that probability into a binary decision:
- If probability > 0.5 → "Fraud"
- If probability ≤ 0.5 → "Legitimate"

The threshold can be adjusted up or down. Section 9 of this notebook is entirely about this.

---

> 💬 **If someone asks "What is machine learning?"**
> *"Instead of writing rules by hand, you give a computer thousands of labelled examples and let it find the patterns itself. For our project: 270,000 labelled transactions in, trained model out, then tested on 68,000 new transactions the model has never seen."*


---

# 1. 🎯 Problem & Motivation

## What are we trying to do?

A credit card company in the western United States gave us records of about **340,000 past purchases**. Each record says where the purchase happened, when, how much it was, who the customer was, and — most importantly — whether it turned out to be fraud or not.

**Our task:** teach a computer to look at a new purchase and decide — *"Should we block this, or let it through?"*

This is called a **classification** problem — the computer has to sort each purchase into one of two groups: **fraud** or **legitimate**. That's it. Two options.

---

## Why is this hard?

Only **0.52%** of all purchases in our data are fraud — roughly **1 out of every 200**.

Now imagine building the laziest possible detector. One that looks at every purchase and always says **"This is fine, let it through."** That lazy detector would technically be right **99.5% of the time** — because 99.5% of purchases genuinely are fine.

But it would **catch zero fraud**. Every criminal gets away.

This is what we call the **accuracy trap** — a high "accuracy" score that sounds impressive but means the detector isn't actually detecting anything. This trap is the single most important thing to understand about this whole project, because it affects every decision we make from here on.

---

## Two types of mistakes

When the detector gets it wrong, there are only two possible ways:

| Mistake | What happens | How serious? |
|---------|-------------|-------------|
| **False alarm** — Detector blocks a real purchase | The customer is annoyed, calls the bank, card gets unblocked in a few minutes | Annoying, but fixable with a phone call |
| **Missed fraud** — Detector lets real fraud through | The criminal takes the money. Customer loses it. The bank has to pay it back | **Serious — the money is gone forever** |

A blocked card can be unblocked in minutes. Stolen money doesn't come back.

That's why the company told us: **"When in doubt, block the purchase. We'd rather annoy a few customers than lose real money."**

---

## How we measure success: Recall

Because missing real fraud is the worst outcome, we focus on a measure called **Recall**. It answers one question:

> *"Out of all the real fraud cases — how many did we actually catch?"*

If there were 100 fraud cases and our detector caught 84 of them, the Recall is **84%**. The higher the number, the more fraud we catch. Everything else in this project is in service of making this number as high as possible.

---

> 💬 **If someone asks "What is this project about?"**
> *"A credit card company gave us 340,000 past transactions. Only 0.5% were fraud. We taught a computer to spot the fraudulent ones. The hard part is that fraud is so rare — a lazy system that says 'everything is fine' would be 99.5% accurate but catch zero fraud."*

---

> 📂 **Where to find the code:** The business context and problem definition are described in the opening markdown of `ML_work_and_analysis.ipynb` — **Cell 1** (the very first cell at the top of the notebook).

---

# 2. 📊 Dataset Overview

*Now we know what we're trying to do — find the 1 fraudulent purchase out of every 200. Next question: what data did we actually have to work with?*

---

## What does the data look like?

Think of a giant spreadsheet. Each **row** is one purchase. Each **column** is one piece of information about that purchase. We had **339,607 rows** and **21 columns** in total.

---

## All 21 original columns — with real examples

| Column | Type | Example value | What it means |
|--------|------|--------------|--------------|
| `trans_date_trans_time` | Timestamp | `2019-01-01 03:24:00` | Exact date and time of the purchase |
| `cc_num` | Number | `3521123456789012` | Card number (anonymised) |
| `merchant` | Text | `fraud_Rippin, Kub and Mann` | Name of the store |
| `category` | Text | `shopping_net` | Type of store (14 categories total) |
| `amt` | Number | `$545.00` | Dollar amount of the purchase |
| `first` / `last` | Text | `Jennifer Banks` | Cardholder's name |
| `gender` | Text | `F` | Cardholder's gender |
| `street` | Text | `351 Harrison Curve` | Cardholder's home street |
| `city` | Text | `Laurel` | Cardholder's home city |
| `state` | Text | `MT` | Cardholder's home US state |
| `zip` | Number | `59833` | Home zip code |
| `lat` / `long` | Decimal | `46.96 / -112.10` | GPS position of the cardholder's **home** |
| `city_pop` | Number | `4154` | Population of the cardholder's home city |
| `job` | Text | `Mechanical engineer` | Cardholder's job title |
| `dob` | Date | `1988-03-09` | Cardholder's date of birth |
| `trans_num` | Text | `2da90c7d74bd46...` | Unique transaction ID |
| `unix_time` | Number | `1325376018` | Transaction time as a Unix timestamp |
| `merch_lat` / `merch_long` | Decimal | `36.01 / -82.05` | GPS position of the **merchant / store** |
| `is_fraud` | 0 or 1 | `0` or `1` | **The label we are trying to predict** |

> **Key observation:** `lat`/`long` is the cardholder's home, and `merch_lat`/`merch_long` is the store. The distance between these two GPS points became one of our most powerful engineered features.

---

## The key numbers

| Fact | Value |
|------|-------|
| Total purchases | **339,607** |
| Fraud cases | **1,782** |
| Fraud rate | **0.52%** (about 1 in 200) |
| Legitimate cases | **337,825** |
| Unique merchant categories | **14** |
| US states covered | **13** (western US focus) |
| Date range | **2019–2020** |
| Missing values | **None** — the dataset is clean |
| Amount range | **$1 → $28,948** |

---

## The 14 merchant categories

Different store types have very different fraud rates. Here are all 14 categories in the dataset:

`shopping_net` · `grocery_pos` · `misc_net` · `gas_transport` · `shopping_pos` · `misc_pos` · `food_dining` · `personal_care` · `home` · `kids_pets` · `health_fitness` · `entertainment` · `travel` · `grocery_net`

**Key finding from our analysis:** Internet-based categories (`shopping_net`, `misc_net`, `grocery_net`) have **significantly higher fraud rates** than physical point-of-sale categories. Online fraud is easier — you only need a stolen card number, not the physical card.

---

## 🔨 What we actually did with this data

When we first loaded the dataset, we didn't jump straight into building models. We spent time getting to know the data first. Here is exactly what we did:

**1. We loaded it and looked at the shape**
First thing — just print how many rows and columns exist. 339,607 rows × 21 columns. This told us the scale of the problem immediately.

**2. We checked every column's data type**
Some columns were numbers, some were text (like `category` and `state`), one was a timestamp, one was a date. This mattered because the computer can only work with numbers — so we already knew which columns would need to be converted later.

**3. We checked for missing values**
We ran a check on every column to see if any values were blank. The result: **zero missing values across all 339,607 rows**. This was good news — no cleaning or filling in gaps was needed.

**4. We looked at summary statistics**
We computed the minimum, maximum, average, and spread for every numeric column. Key discovery here: the `amt` column ranged from $1 to $28,948 — a massive spread. This immediately told us we would need to handle extreme values carefully later (which we did with log-transform).

**5. We confirmed the fraud rate**
We counted how many rows had `is_fraud = 1` vs `is_fraud = 0`. Result: 1,782 fraud cases out of 339,607 total. That's 0.52%. This single number defined the entire challenge of the project.

**6. We noted which columns would be useful vs which were noise**
- `trans_date_trans_time` — useful, but needs to be broken apart into hour/day/month
- `lat`, `long`, `merch_lat`, `merch_long` — four GPS columns that together could give us distance
- `dob` — useful only if we convert it to age
- `merchant`, `job`, `trans_num` — too many unique values, unlikely to generalise

All of these decisions fed directly into the next section — Feature Engineering.

---

## Why these numbers matter

Out of nearly 340,000 purchases, only **1,782 are fraud**. Imagine you are airport security. For every 200 passengers who walk through, only **1** is carrying something suspicious. Your job is to find that 1 person — without making the other 199 miss their flights.

This extreme gap is called **class imbalance**, and it's the central challenge of the entire project. Almost every decision we make from this point forward is shaped by it.

---

> 💬 **If someone asks "What data did you use?"**
> *"A public dataset of 340,000 credit card transactions from the western US, covering 2019–2020. Each transaction has 21 columns — time, amount, GPS location, store type, cardholder details — plus a fraud label. Only 0.52% were fraud. The first thing we did was check the data types, confirm no missing values, and look at the distribution of amounts and the fraud rate."*

---

> 📂 **Where to find the code:**
> - **Loading the dataset** → `ML_work_and_analysis.ipynb`, **Cells 2–8** (Section 1: Data Loading & Initial Inspection)
> - The code downloads data via `kagglehub`, shows column types, checks for missing values, and prints summary statistics.


---

# 3. 🔍 Exploratory Data Analysis (EDA)

*We have the data. But before building anything, we need to actually look at it and understand: what does fraud look like? Is it different from normal purchases in any obvious way?*

---

## What is EDA?

**Exploratory Data Analysis** means studying the data through charts and numbers before building any models. Think of a detective studying evidence before starting an investigation — you need to understand the patterns before you can tell the computer what to look for.

This step directly shapes what features we create in Section 4 and what the model ends up learning.

---

## 🔨 What we actually did

We did not just glance at the data — we made **7 separate visualisations**, each answering a specific question. Here is what we built and what we found:

**Chart 1 — Bar chart + donut chart of fraud vs legitimate**
We plotted how many transactions were fraud and how many were legitimate. Result: the donut chart showed 99.48% legitimate, 0.52% fraud. Seeing it visually made the imbalance feel real in a way that just reading "0.52%" does not.

**Chart 2 — Fraud rate by hour of day (bar chart)**
We grouped all 339,607 transactions by the hour they occurred (0–23) and calculated the fraud rate for each hour. The pattern was striking: fraud spikes sharply between 1 AM and 4 AM and drops to near zero during business hours. This directly told us: we must extract `hour` as a feature.

**Chart 3 — Fraud rate by day of week**
Same idea but for Monday–Sunday. The pattern was less extreme than hours, but there were small differences between weekdays and weekends, so we included `day_of_week` as a feature too.

**Chart 4 — Transaction amount: histogram + box plot, fraud vs legitimate**
We plotted the distribution of `amt` for fraud and legitimate separately. The fraud distribution had a much longer right tail — many more high-value transactions. Average fraud amount ($518) was over 7× the legitimate average (~$70). This confirmed `amt` would be a strong feature. The extreme range also showed we needed `log_amt` to compress it.

**Chart 5 — Fraud rate by merchant category (horizontal bar chart)**
We calculated the fraud rate for each of the 14 store categories. Online categories (`shopping_net`, `misc_net`, `grocery_net`) had the highest rates. Physical categories like `gas_transport` and `food_dining` had very low rates. This told us: `category` is a useful signal — we need to encode it for the model.

**Chart 6 — Fraud rate by US state + geographic scatter plot**
We plotted fraud by state (top 15) and drew a scatter plot of all 5,000 sampled transactions on a map (longitude vs latitude, coloured by fraud). The scatter plot showed that fraud cases appeared spread across the country — not concentrated near cardholders' homes. This was the first hint that distance from home would be important.

**Chart 7 — Correlation heatmap**
We calculated how strongly each numeric column correlated with `is_fraud`. The result: all correlations were weak (between -0.3 and +0.3). This told us there was no single magic column that predicts fraud on its own. Fraud is defined by combinations — exactly why simple rule-based systems fail and why we need ML.

**Bonus — Violin and box plots**
We compared the distributions of `amt`, `city_pop`, `lat`, and `long` between fraud and legitimate for a visual sanity check. Confirmed that amounts were the most visually different between the two classes.

---

## What we discovered: five signals that give fraud away

### 🕐 Signal 1: Fraud happens at night

Most fraud happens **between 1 AM and 4 AM**. The fraud rate at 2–3 AM is nearly **3× the daily average**. The reason is simple: if someone steals your card details, they want to use them while you are asleep and not watching your banking app. By the time you wake up and notice, the damage is done.

**What we did with this:** We extracted `hour` as a new feature in Section 4. It became one of the **top 5 most important clues** in the final model.

---

### 💰 Signal 2: Fraudsters spend unusually large amounts

| | Average amount | Median amount |
|---|---|---|
| **Fraud** | **$518** | ~$380 |
| **Legitimate** | **~$70** | ~$47 |

Criminals go for expensive items — electronics, luxury goods — because they want to steal as much as possible before the card gets blocked.

**What we did with this:** We kept `amt` as a feature and also created `log_amt` (the compressed version) to handle the extreme $1–$28,948 range.

---

### 🛒 Signal 3: Online store categories are the riskiest

| Rank | Category | Type |
|------|---------|------|
| 1 | `shopping_net` | Online shopping |
| 2 | `grocery_net` | Online grocery |
| 3 | `misc_net` | Misc internet purchases |
| 4 | `shopping_pos` | In-store shopping |
| 5+ | Physical point-of-sale categories | Much lower fraud rate |

When you buy in a physical store, you need the actual card, a PIN, and your face may be on camera. Online, only the card number is needed — and stolen numbers are trivially easy to use.

**What we did with this:** We one-hot encoded `category` in Section 4 — converting the text into 14 yes/no columns the model can read.

---

### 📍 Signal 4: The purchase is far from home

Fraudulent purchases tend to happen at stores that are **geographically far** from where the cardholder lives. A card used thousands of kilometres from the owner's home city is extremely suspicious.

**What we did with this:** We calculated the exact distance between the cardholder's home GPS and the merchant's GPS using the Haversine formula (see Section 4). This became one of the **top 3 most important clues** in our final model — and it was not in the original data at all.

---

### 📊 Signal 5: No single feature is enough — combinations are what matter

The correlation heatmap showed that no single column has a strong linear relationship with `is_fraud`. All individual correlations were weak (+/- 0.1 to 0.3). This told us:

- **No simple rule exists** — you can't say "if amount > $X, it's fraud"
- **Combinations are what matter** — fraud = large amount AND late night AND far from home AND online store
- **This is why we need ML** — a machine learning model can learn these combinations automatically; a set of manual rules cannot

---

## Why this step mattered for everything that followed

Every chart we made in this section directly informed a decision in the next section. The hour chart → we extract `hour`. The amount distribution → we create `log_amt`. The category chart → we encode `category`. The geographic scatter → we calculate `distance_km`. None of the feature engineering in Section 4 was guesswork — it all came from what we saw here.

---

> 💬 **If someone asks "How does fraud look different from normal purchases?"**
> *"We made 7 visualisations. The clearest findings: fraud peaks between 1–4 AM (3× the average rate), involves much larger amounts ($518 vs $70), hits online stores hardest, and happens far from the cardholder's home. Each of these observations became a feature we gave to the model. The correlation heatmap also showed that no single column alone predicts fraud — it's always a combination."*

---

> 📂 **Where to find the code:**
> - **Class balance charts** → `ML_work_and_analysis.ipynb`, **Cell 10**
> - **Fraud rate by hour and day** → **Cell 13**
> - **Transaction amount distribution** → **Cell 16**
> - **Fraud rate by merchant category** → **Cell 19**
> - **Geographic scatter + state analysis** → **Cell 22**
> - **Violin & box plots by class** → **Cell 25**
> - **Correlation heatmap** → **Cell 28**


---

# 4. ⚙️ Feature Engineering

*We have five fraud signals. Now we need to put those signals into a form the computer can actually learn from.*

---

## What is feature engineering?

A **feature** is one piece of information the computer uses — one column in the spreadsheet. **Feature engineering** means creating new, smarter columns from the data we already have.

The raw data has a timestamp like `"2019-01-01 03:24:00"`. To a human, "3:24 AM" screams suspicious. A computer cannot read that text — but if we **pull out just the number 3** (the hour), the computer can learn that hour=3 correlates with fraud.

We are not inventing information that doesn't exist. We are **reshaping what we already have** so the computer can understand it.

---

## 🔨 What we actually did — step by step

After the EDA section gave us clear signals, we wrote a single Python function that transformed the raw dataset. Here is exactly what we did, in order:

**Step 1 — Parsed the timestamp into three new columns**
The raw column `trans_date_trans_time` contained strings like `"2019-01-01 03:24:00"`. We used Python's `pandas` library to extract:
- `hour` — the hour (0–23)
- `day_of_week` — the day of the week (0=Monday, 6=Sunday)
- `month` — the month (1–12)

The original timestamp column was then **deleted** — its information was now captured in three usable numbers.

**Step 2 — Calculated the customer's age**
The `dob` column had dates of birth like `1988-03-09`. We subtracted the date of birth from the transaction date to get the customer's age in years at the time of purchase. The original `dob` column was then **deleted**.

**Step 3 — Calculated the distance between home and store**
This was the most important step. The dataset had four GPS columns: `lat`/`long` (cardholder's home) and `merch_lat`/`merch_long` (the merchant). We applied the **Haversine formula** to calculate the straight-line distance in kilometres between them for every one of the 339,607 transactions. This produced a new column `distance_km`. The four GPS columns were kept in the model as well.

**Step 4 — Applied a log-transform to the amount**
Purchase amounts ranged from $1 to $28,948. That extreme range causes problems — a $28,000 transaction would dominate every calculation just because of its size. We created `log_amt = log(amt + 1)`, which compresses the range while preserving the ordering. A $28,000 purchase still scores higher than a $50 purchase — just not by a factor of 560.

**Step 5 — One-hot encoded `category` and `state`**
The `category` column had 14 possible text values. The `state` column had 13 possible text values. We converted each into a set of yes/no columns. So `category = "shopping_net"` became a `1` in the `cat_shopping_net` column and `0` in all others.

**Step 6 — Dropped the columns that were useless**
`merchant` (hundreds of unique store names — too random), `job` (hundreds of unique job titles), `trans_num` (random ID), `city` (already covered by GPS), and personal identifiers (`first`, `last`, `street`, `zip`, `cc_num`) were all removed.

**The result:** We went from 15 useful raw columns to **39 numeric columns** — all numbers, all meaningful, all ready to be fed into a model.

---

## The 6 new columns we created

| New column | What it is | Where it comes from | Why it matters |
|-----------|-----------|-----------|----------------------|
| **`hour`** | Hour of day (0–23) | Extracted from timestamp | Fraud peaks at 1–4 AM — our top EDA finding |
| **`day_of_week`** | Day number (0=Mon … 6=Sun) | Extracted from timestamp | Weekend vs weekday patterns |
| **`month`** | Month number (1–12) | Extracted from timestamp | Seasonal variation |
| **`age`** | Customer's age in years at purchase time | `(transaction_date − date_of_birth) ÷ 365` | Certain age groups are targeted more |
| **`distance_km`** | Distance in km between home and store | **Haversine formula** on GPS coordinates | Far-from-home = highly suspicious. **Top 3 feature** |
| **`log_amt`** | Natural log of (amount + 1) | `log(amt + 1)` | Compresses $1–$28,948 range; stops extreme values dominating |

---

## How we calculated distance — the Haversine formula

The **Haversine formula** computes the shortest distance between two GPS points on the surface of a sphere (the Earth). In plain language:

> *Take the customer's home GPS position (lat/long). Take the merchant's GPS position (merch_lat/merch_long). Calculate the straight-line distance in kilometres between them.*

A purchase 2,000 km from the cardholder's home city is far more suspicious than one 5 km away. This column was **not in the original data** — we created it from four GPS columns that were already there.

```python
def haversine_approx(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * arctan2(sqrt(a), sqrt(1-a))

df['distance_km'] = haversine_approx(df['lat'], df['long'], df['merch_lat'], df['merch_long'])
```

---

## Turning text into numbers — One-Hot Encoding

The computer can only work with numbers. Some columns contain text — the store category (`"shopping_net"`, `"grocery_pos"`) and the US state (`"MT"`, `"CA"`).

**One-hot encoding** creates a new yes/no column for each possible text value (1 = yes, 0 = no):

| Original category | → `cat_shopping_net` | → `cat_grocery_pos` | → `cat_misc_net` | ... |
|------------------|---|---|---|---|
| `shopping_net` | **1** | 0 | 0 | ... |
| `grocery_pos` | 0 | **1** | 0 | ... |
| `misc_net` | 0 | 0 | **1** | ... |

This was applied to:
- **14 merchant categories** → 14 new yes/no columns
- **13 US states** → 13 new yes/no columns
- Total: **27 new columns** just from text encoding

---

## Columns we dropped — and why

| Dropped column | Reason |
|---------------|--------|
| `trans_date_trans_time` | Replaced by `hour`, `day_of_week`, `month` |
| `dob` | Replaced by `age` |
| `merchant` | Hundreds of unique store names — too random to generalise |
| `city` | Already captured by `lat`/`long` GPS coordinates |
| `trans_num` | Random transaction ID — no pattern, just noise |
| `job` | Hundreds of unique job titles — no meaningful signal |
| `cc_num`, `first`, `last`, `street`, `zip` | Personal identifiers — no modelling value |

---

## The before and after

| | Before engineering | After engineering |
|---|---|---|
| **Number of columns** | 15 usable | **39 usable** |
| **Column types** | Mix of text and numbers | **All numbers** |
| **EDA signals encoded** | None explicitly | All 4 fraud signals are now features |

> **4 of the top 5 most important features** in the final model were either created by us (`distance_km`, `hour`, `age`, `log_amt`) or encoded from text by us (category columns). Feature engineering directly made the model better.

---

> 💬 **If someone asks "What is feature engineering?"**
> *"The raw data had timestamps and GPS coordinates the computer couldn't use directly. We split the timestamp into hour, day, and month. We calculated the distance between the cardholder's home and the store using the four GPS columns. We computed age from date of birth. We converted store categories from text to yes/no columns. In the end we went from 15 messy columns to 39 clean numeric ones — and 4 of our handmade features ended up in the top 5 most important signals the model used."*

---

> 📂 **Where to find the code:**
> - **Strategy table** → `ML_work_and_analysis.ipynb`, **Cell 30** (Section 3 markdown)
> - **Feature engineering code** → **Cell 31** (creates all 6 new columns, one-hot encoding, drops unused columns)
> - **Verification** → **Cell 32** (prints final column list: 39 columns, all numeric)


---

# 5. 🔪 Data Preparation

*We have 39 clean, numeric columns. Before the computer can learn anything, we need to do three more things with the data. Getting any of these wrong is one of the most common ways to build something that looks great on paper but fails in reality.*

---

## 🔨 What we actually did — the three steps in practice

**Step 1 — We split the 339,607 rows into two sealed groups**

We used scikit-learn's `train_test_split` with `test_size=0.2` and `stratify=y`. This means:
- 80% of rows (≈271,685) went into the **training set** — this is what the models learn from
- 20% of rows (≈67,922) went into the **test set** — this was immediately sealed and not touched again until the very final evaluation

The `stratify=y` parameter ensured that both groups preserved the exact 0.52% fraud rate. Without this, random chance could have put most fraud cases in one group, making training or testing unreliable.

We also set a fixed `random_state=42`. This means anyone running our code gets the exact same split — full reproducibility.

**Step 2 — We scaled the numeric columns (fitted on training data only)**

We used scikit-learn's `StandardScaler`. This shifts every numeric column to have a mean of 0 and a standard deviation of 1 after transformation. 

Critically: we **fitted** the scaler only on the training set, then **applied** it to both training and test sets. We never let the scaler see the test data during fitting. This prevents data leakage — a mistake where information from the "exam" creeps into the "study phase."

**Step 3 — We balanced the training data with SMOTE (training set only)**

We used `SMOTE` from the `imbalanced-learn` library. SMOTE went through the training fraud cases and synthetically created new ones by interpolating between real examples — until fraud and legitimate were 50/50.

Before SMOTE: ~1,407 fraud cases (0.52%) and ~270,278 legitimate cases.
After SMOTE: ~270,278 fraud cases and ~270,278 legitimate cases — a balanced dataset.

**The test set was never touched by SMOTE.** It stayed at the real 0.52% fraud rate, because that is what the model will face in the real world. If we had balanced the test set too, the scores would have been artificially inflated.

---

## Step 1: Split the data into "study notes" and "the exam"

We divided all 339,607 purchases into two separate groups:

| Group | How much | What it's for |
|-------|---------|---------|
| **Training set** (80%) | ~271,685 purchases | What the computer **learns from** — these are the study notes |
| **Test set** (20%) | ~67,922 purchases | The **final exam** — the computer has never seen these and never will until the very end |

**Why split?** Think of school. You study from your notes, then you prove what you learned on the exam. If the exam contained the exact same questions you practised on, your grade would mean nothing — you'd just be repeating memorised answers, not proving you actually learned anything.

Two important details:
- Both groups have the **same 0.52% fraud rate** — stratified split
- **Fixed random seed** — anyone running the code gets the exact same split

---

## Step 2: Put all columns on equal footing (scaling)

Different columns have wildly different number ranges:
- **City population** goes from 0 to 2,000,000
- **Hour of day** goes from 0 to 23
- **Purchase amount** goes from $1 to $28,000

Some learning methods treat **bigger numbers as more important**. That means "city population" (millions) would completely overpower "hour of day" (0–23), even though the hour is actually a much better fraud clue. That's not fair.

**Scaling** adjusts all columns to the same range — centred around 0. After scaling, every column gets a fair chance to contribute.

### The rule we followed strictly

> We **calculated the scaling only from the training data**. Then we applied that same scaling to the test data. If we had peeked at the test data first, information from the "exam" would have leaked into the "study" phase — this is called **data leakage** and it makes results look better than they really are.

---

## Step 3: Balance the training data (SMOTE)

Remember: only 0.52% of purchases are fraud. If the computer trains on this as-is, it learns a lazy shortcut: **"just say 'not fraud' every time."** That's the accuracy trap from Section 1.

**SMOTE** creates **new, realistic-looking fake fraud examples** and adds them to the training set until fraud and legitimate are evenly balanced:

| | Before SMOTE | After SMOTE |
|---|---|---|
| Legitimate | ~270,278 | ~270,278 (unchanged) |
| Fraud | ~1,407 (0.52%) | **~270,278 (50%)** |

**How does SMOTE create new fraud examples?** It takes a real fraud case, finds the most similar real fraud cases near it, and creates a brand new point somewhere in between. The result looks like a realistic fraud purchase — similar to real ones, but not an identical copy.

### The rule we followed strictly

> SMOTE is **only applied to the training data**. The test set stays at the real 0.52% — because that's what the model will face in the real world.

---

## Why this section matters

These three steps — splitting, scaling, and balancing — are invisible to most people. But getting them wrong is **the number one beginner mistake** in machine learning. Our project followed every best practice: no data leakage, no touching the test set, stratified split, scaling from training data only, SMOTE on training data only.

---

> 💬 **If someone asks "How did you prepare the data?"**
> *"We split 80/20 with stratification so both parts kept the same fraud rate. We scaled all 39 columns using only the training data — the test set was never touched during scaling. Then we used SMOTE on the training set only to balance fraud vs legitimate from 0.52%/99.48% to 50%/50%. The test set stayed at the real 0.52% throughout."*

---

> 📂 **Where to find the code:**
> - **Explanation of the three steps** → `ML_work_and_analysis.ipynb`, **Cell 33**
> - **The actual code** → **Cell 34** (80/20 split, StandardScaler, SMOTE)


---

# 6. 📏 Evaluation Metrics

*The data is ready. Before we train any models, we need to agree on **how we'll judge them**. This is important — because the obvious measure (accuracy) is completely useless here, as we saw in Section 1.*

---

## Why "accuracy" fails us

Quick reminder: a detector that always says "not fraud" would be **99.5% accurate** — and catch zero fraud. Accuracy tells us how often the model is right **overall**, but since 99.5% of purchases are legitimate, the model gets a near-perfect score just by getting the easy majority right. It tells us nothing about what really matters: **what happened to the fraud cases**.

We need measurements that focus specifically on the **1-in-200 fraud transactions**.

---

## The four measurements we use

### 🎯 1. Recall — "How much fraud did we catch?"

This is our **most important measure**. It answers:

> *"Out of all the real fraud cases, how many did the model find?"*

**Concrete example from our project:** There are 356 fraud cases in the test set. Our best model caught 299 of them.

**Recall = 299 ÷ 356 = 84%**

The higher this number, the more fraud we catch. This is what the business cares about above all else.

---

### 🔍 2. Precision — "How many of our alerts were real?"

This answers the opposite question:

> *"Of everything we flagged as fraud, how much was actually fraud?"*

**Concrete example:** The model flagged 323 purchases. Of those, 299 were real fraud and 24 were innocent purchases that got wrongly blocked.

**Precision = 299 ÷ 323 = 93%**

The higher this number, the fewer angry customers whose cards were wrongly blocked.

---

### ⚖️ 3. F1-Score — "The balance between the two"

Here's the problem with focusing on just one measure. You could flag **every single purchase** as fraud. You'd catch 100% of the fraud (perfect Recall!) — but you'd also block every legitimate customer (terrible Precision).

The **F1-Score** is a single number that rewards you **only when both Recall and Precision are decent**. It punishes you for being extreme in either direction. Think of it as a fairness judge that makes sure you're not gaming the system.

---

### 📈 4. ROC-AUC — "Overall quality grade"

This measures how well the model can tell fraud apart from legitimate **overall**, across all possible settings. Think of it like a school grade:

| AUC Score | Grade |
|-----------|-------|
| **0.5** | Random guessing — coin flip |
| **0.8** | Good |
| **0.9** | Excellent |
| **1.0** | Perfect — never wrong |

Our best model scored **0.998** — nearly perfect.

---

## The seesaw trade-off

There is always a trade-off:

- **Catch more fraud** (higher Recall) → you'll also raise more false alarms (lower Precision)
- **Fewer false alarms** (higher Precision) → you'll miss more fraud (lower Recall)

It's like a seesaw — pushing one side up pushes the other down. You can never have both at 100%. Our business said **"catch as much fraud as possible"**, so we lean toward higher Recall and accept a few more false alarms.

---

> 💬 **If someone asks "How do you measure if the model is good?"**
> *"We don't use accuracy because it's misleading — even a useless model gets 99.5%. Instead, we use Recall (how much fraud we catch), Precision (how many of our alerts are real), and F1-Score (the balance between the two). Our top model catches 84% of fraud with only 24 false alarms out of 68,000 purchases."*

---

> 📂 **Where to find the code:**
> - **Detailed metric explanations** → `ML_work_and_analysis.ipynb`, **Cell 48** (Section 6 markdown explaining all five metrics, the trade-off, and what each visualisation shows)
> - **Metric calculations for all models** → **Cell 49** (computes Accuracy, Precision, Recall, F1, ROC-AUC for all 5 models on the test set)

---

# 7. 🤖 Model Training

*We have clean data, smart features, a balanced training set, and clear measures of success. Now it's time to actually teach the computer to detect fraud. We tried five approaches — from the simplest to the most advanced.*

---

## What does "training a model" mean?

**Training** means showing the computer thousands of labelled examples — "this purchase was fraud, this one wasn't" — and letting it figure out the patterns on its own.

Imagine teaching a small child to tell dogs from cats. You don't give them a rulebook. You show them hundreds of photos: "dog, cat, dog, cat..." Eventually they learn what makes a dog look different from a cat — not because you explained the rules, but because they saw **enough examples**. Machine learning is the same idea, but with numbers and "fraud vs legitimate."

---

## 🔨 What we actually did — how we trained and tuned all five models

We did not just run each model once with default settings and pick a winner. We did this properly:

**1. We set up an automated search for the best settings (RandomizedSearchCV)**
Every model has settings called **hyperparameters** — things like "how many trees to build" or "how deep each tree can grow." These are chosen before training and they matter a lot. Wrong settings = bad model, even with a good algorithm.

Instead of guessing, we wrote code that **automatically tried 10–20 different combinations** of settings for each model. For each combination, it tested performance using **5-fold cross-validation** (the training data is split into 5 equal parts; the model is trained on 4 and tested on 1, rotated 5 times). The combination with the best average score across all 5 folds was kept.

**2. We trained each model using its best settings**
Once the search found the best settings, we retrained each model on the full training set with those settings locked in.

**3. We kept the test set completely untouched during all of this**
The 67,922 test transactions were never used during training, tuning, or cross-validation. They stayed sealed until Section 8.

**4. We noted what the best settings were for each model**

| Model | Key settings found |
|-------|-------------------|
| Logistic Regression | `C=0.01`, `class_weight='balanced'` |
| Decision Tree | `max_depth=8`, `min_samples_leaf=20` |
| Random Forest | `n_estimators=200`, `max_depth=15` |
| XGBoost | `n_estimators=300`, `learning_rate=0.05`, `scale_pos_weight=190` |
| LightGBM | `n_estimators=500`, `num_leaves=63`, `learning_rate=0.05` |

Note: `class_weight='balanced'` and `scale_pos_weight=190` (for XGBoost) are how we told the models to pay extra attention to the rare fraud class even during training — another safeguard against the accuracy trap.

---

## The 5 models we trained (simplest → most advanced)

We deliberately started simple and worked upward. This way we could measure exactly how much each step of added complexity helps.

---

### Model 1: Logistic Regression — "The Straight Line"

**How it works:** Draws a single straight boundary through the data. Everything on one side = fraud. Everything on the other side = legitimate.

**Analogy:** Drawing one straight line on a map to separate two neighbourhoods. Works fine when the boundary is straight — fails when it's curvy.

- **Strength:** Very fast, very interpretable — you can read the formula
- **Weakness:** Fraud is defined by *combinations* ("high amount AND 3 AM AND far from home"). A single straight line can't capture that
- **Role:** Our **baseline** — the minimum standard every other model must beat
- **Best settings found:** `C=0.01`, `class_weight='balanced'`, `max_iter=1000`

---

### Model 2: Decision Tree — "The Flowchart"

**How it works:** Asks a chain of yes/no questions — *"Amount > $500? → Yes → Hour < 5? → Yes → Distance > 500 km? → Yes → Fraud."*

**Analogy:** The game of "20 Questions" — each question narrows down the answer until you reach a conclusion.

- **Strength:** Can handle complex combinations — not limited to a straight line
- **Weakness:** Prone to **overfitting** — memorises training examples instead of learning general patterns
- **Best settings found:** `max_depth=8`, `min_samples_leaf=20`, `class_weight='balanced'`

---

### Model 3: Random Forest — "Crowd Wisdom"

**How it works:** Builds **hundreds of flowcharts**, each using a slightly different random sample of data and columns. All trees **vote** — majority wins.

**Analogy:** Instead of asking one doctor, you ask 200 doctors who each read slightly different textbooks. The crowd is more reliable than any single expert.

- **Strength:** Individual tree mistakes cancel each other out — much more stable
- **Weakness:** The 200 trees work independently — they don't learn from each other's mistakes
- **Best settings found:** `n_estimators=200`, `max_depth=15`, `min_samples_leaf=5`, `class_weight='balanced'`

---

### Model 4: XGBoost — "Learning from Mistakes"

**How it works:** Builds trees **sequentially**. Each new tree specifically focuses on the purchases the previous trees got wrong. Tree 1 makes errors. Tree 2 corrects them. Tree 3 corrects what Tree 2 missed. Continues for hundreds of rounds.

**Analogy:** A team of editors reviewing a document — each round focuses on what the previous round missed. After 200 rounds, nearly error-free.

- **Strength:** Catches the most raw fraud cases of all five models
- **Weakness:** More false alarms than LightGBM (47 vs 24)
- **Best settings found:** `n_estimators=300`, `max_depth=6`, `learning_rate=0.05`, `subsample=0.8`, `scale_pos_weight=190`

---

### Model 5: LightGBM — "The Smart Editor" *(our winner)*

**How it works:** Same sequential "learn from mistakes" idea as XGBoost, but grows trees **leaf-first** — focuses computation on the parts of the data most likely to reduce errors. Significantly faster on large datasets. Created by Microsoft.

**Analogy:** Same team of editors, but they zero in on the paragraphs most likely to still contain errors rather than re-reading everything every time.

- **Strength:** Best balance of fraud caught vs false alarms
- **Strength:** Trains much faster — critical at production scale
- **Best settings found:** `n_estimators=500`, `num_leaves=63`, `learning_rate=0.05`, `min_child_samples=20`, `class_weight='balanced'`
- **This is the model we deployed** in the live demo

---

## Hyperparameter search summary

| Model | Combinations tested | CV folds | Total model fits |
|-------|-------------------|----------|-----------------|
| Logistic Regression | 10 | 5 | 50 |
| Decision Tree | 20 | 5 | 100 |
| Random Forest | 20 | 5 | 100 |
| XGBoost | 20 | 5 | 100 |
| LightGBM | 20 | 5 | 100 |

---

> 💬 **If someone asks "What models did you try?"**
> *"Five models from simple to advanced. We didn't just run them with default settings — we used automated search with cross-validation to find the best settings for each one. LightGBM won: 84% fraud caught, only 24 false alarms per 68,000 transactions."*

---

> 📂 **Where to find the code:**
> - **Model overview** → `ML_work_and_analysis.ipynb`, **Cell 35**
> - **Logistic Regression** → **Cell 37** · **Decision Tree** → **Cell 39**
> - **Random Forest** → **Cell 41** · **XGBoost** → **Cell 43** · **LightGBM** → **Cell 45**
> - **Best hyperparameter summary table** → **Cell 47**


---

# 8. 📈 Model Results

*All five models have been trained. Now comes the moment of truth: we run each model against the 67,922 purchases it has never seen and see who actually learned the patterns.*

---

## 🔨 What we actually did — how we evaluated the models

**1. We unsealed the test set for the first time**
The 67,922 test transactions had been locked away since the split in Section 5. We now fed them through each of the five trained models — no looking before this point.

**2. We computed five metrics for each model**
For every model, we calculated: Accuracy, Precision, Recall, F1-Score, and ROC-AUC. We deliberately **did not look at accuracy** when comparing models — we focused on Recall and F1-Score, as explained in Section 6.

**3. We built a colour-coded comparison table**
We put all five models side by side in a table with the key metrics, highlighted the winner in green, and noted where each model fell short.

**4. We drew ROC curves and Precision-Recall curves**
These are visualisations that show model performance across all possible threshold settings (not just the default 0.5). The area under the ROC curve (ROC-AUC) summarises this into one number. LightGBM scored 0.998 — nearly the maximum of 1.0.

**5. We drew confusion matrices for the best and worst models**
A confusion matrix shows exactly what happened to every single test purchase: how many fraud cases were caught, how many slipped through, how many real purchases were wrongly blocked. We printed one for LightGBM (best) and one for Logistic Regression (worst) to show the contrast.

**6. We calculated the dollar impact of each model**
We took the number of missed fraud cases × $518 (average fraud amount) to compute the estimated financial loss from each model. This made the performance differences feel real and comparable.

**7. We extracted feature importance from LightGBM**
After training, LightGBM can tell us which features it relied on most — essentially, which columns were most useful in its decision-making. We plotted these as a bar chart. The top features confirmed that our handmade columns (`distance_km`, `hour`, `age`, `log_amt`) were genuinely the most important signals.

---

## The scoreboard

| Model | Recall | F1-Score | ROC-AUC | Fraud Caught (of 356) | False Alarms |
|-------|--------|----------|---------|----------------------|-------------|
| **LightGBM** ★ | **84.0%** | **88.1%** | **0.998** | **299** | **24** |
| XGBoost | 87.6% | 87.3% | 0.996 | 312 | 47 |
| Random Forest | 70.2% | 76.8% | 0.990 | 250 | 50 |
| Decision Tree | 77.0% | 69.5% | 0.884 | 274 | 130 |
| Logistic Regression | 66.6% | 38.4% | 0.877 | 237 | 641 |

*(Accuracy omitted — it ranges 98.9%–99.9% across all models and tells us nothing useful on imbalanced data.)*

---

## Reading the scoreboard

### The "learn from mistakes" models dominate
XGBoost and LightGBM beat every other approach on all meaningful measures. Sequential, self-correcting chains of trees are clearly the most effective strategy.

### XGBoost catches the most raw fraud — but at a cost
XGBoost caught 312 of 356 fraud cases. But it wrongly blocked **47 legitimate purchases** — nearly double LightGBM's 24 false alarms. Every false alarm means a real customer whose card gets declined.

### LightGBM has the best balance — that's why we chose it
LightGBM caught 299 fraud cases with only 24 false alarms — the highest F1-Score (88.1%) of all five models.

### Simple models fail badly
Logistic Regression: 237 fraud caught, **641 false alarms** — more wrong blocks than actual catches. Not deployable.

---

## The Confusion Matrix — what happened to every purchase

For LightGBM on 67,922 test purchases:

| | Model said **"FRAUD"** | Model said **"LEGITIMATE"** |
|---|---|---|
| **Actually FRAUD** (356 total) | ✅ **299** caught | ❌ **57** slipped through |
| **Actually LEGITIMATE** (67,566 total) | ⚠️ **24** wrongly blocked | ✅ **67,542** correctly approved |

- **299 wins** — criminals stopped. At $518 average, that's **~$154,882 protected**
- **57 misses** — fraud that got through, typically small amounts, daytime, close to home — the hardest cases
- **24 false alarms** — real customers with a briefly declined card. Resolved with a phone call
- **67,542 smooth approvals** — the vast majority of real customers see nothing unusual

---

## What clues did LightGBM rely on most?

| Rank | Feature | Created by us? | Why it matters |
|------|---------|---------------|----------------|
| 1 | `amt` (transaction amount) | No — original data | Fraud amounts are 7× higher on average |
| 2 | `log_amt` (compressed amount) | **Yes** | Same signal, better numerical shape for the model |
| 3 | `distance_km` (home-to-merchant) | **Yes** | Purchases far from home are highly suspicious |
| 4 | `age` (cardholder age) | **Yes** | Certain age groups disproportionately targeted |
| 5 | `hour` (hour of day) | **Yes** | Fraud peaks 1–4 AM |
| 6+ | Category and state columns | Encoded by us | Online categories have much higher fraud rates |

**4 of the top 5 features were created or encoded by us.** This directly proves that EDA + feature engineering — not just picking a powerful algorithm — made the model accurate.

---

## Why did the model miss 57 fraud cases?

The 57 missed cases were the hardest to detect — they looked almost like normal purchases:

- **Small amounts** — close to the $70 legitimate average, so the amount signal was weak
- **Normal hours** — daytime transactions, no "3 AM" red flag
- **Close to home** — geographically nearby, no distance signal
- **Physical store categories** — not the high-risk online categories

These are exactly the borderline cases a real-world fraud team would review manually. The model catches the obvious cases automatically, and the ambiguous ones get human attention.

---

> 💬 **If someone asks "Which model is best and why?"**
> *"LightGBM. We ran all five models on 67,922 transactions they had never seen. LightGBM caught 299 of 356 fraud cases with only 24 false alarms. XGBoost caught more fraud but blocked twice as many real customers. We also found that the features we engineered ourselves — distance from home, hour of day, customer age — were the most important signals the model used."*

---

> 📂 **Where to find the code:**
> - **All model metrics** → `ML_work_and_analysis.ipynb`, **Cell 49**
> - **Comparison table** → **Cell 50** · **Winner announcement** → **Cell 51**
> - **Bar charts** → **Cell 52** · **ROC curves** → **Cell 54** · **Precision-Recall curves** → **Cell 56**
> - **Confusion matrices** → **Cell 58** · **Business impact** → **Cell 60** · **Feature importance** → **Cells 63 & 65**


---

# 9. 🎚️ Threshold Optimisation

*We've picked LightGBM as our winner. But there's one more dial we can turn to fine-tune how sensitive it is. This is where the business gets to make a choice.*

---

## How the model actually makes a decision

When the model looks at a purchase, it doesn't simply say "fraud" or "not fraud." It produces a **confidence number** between 0 and 1:

- *"I'm 92% sure this is fraud"* → 0.92
- *"I'm 15% sure this is fraud"* → 0.15
- *"I'm 50% sure this is fraud"* → 0.50

We then need a **cutoff line** (called the threshold) to turn that number into a yes/no decision. The default cutoff is **0.5** — if the model is more than 50% confident, it says "fraud." Otherwise, "legitimate."

But 0.5 is just a starting point. We can move it up or down.

---

## The smoke alarm analogy

| Setting | What it's like | What happens |
|---------|-------------|-------------|
| **Low threshold (0.3)** | A very sensitive smoke alarm — goes off at the slightest whiff | Catches more fraud, but blocks more real purchases too |
| **Default (0.5)** | Normal sensitivity | Balanced |
| **High threshold (0.7)** | Only goes off for thick smoke or real fire | Fewer false alarms, but some real fraud gets through |

---

## What we tested

We ran LightGBM at different threshold settings and measured the results:

| Threshold | Fraud Caught (of 356) | False Alarms | Recall | Precision | F1-Score |
|-----------|----------------------|-------------|--------|-----------|----------|
| 0.1 (very sensitive) | 310 | 45 | 87.1% | 87.3% | 87.2% |
| 0.2 | 309 | 36 | 86.8% | 89.6% | 88.2% |
| 0.3 | 306 | 33 | 86.0% | 90.3% | 88.1% |
| **0.4 (sweet spot)** ★ | **304** | **28** | **85.4%** | **91.6%** | **88.4%** |
| 0.5 (default) | 299 | 24 | 84.0% | 92.6% | 88.1% |
| 0.6 | 295 | 22 | 82.9% | 93.1% | 87.7% |
| 0.7 (cautious) | 293 | 18 | 82.3% | 94.2% | 87.9% |

**The sweet spot is 0.4.** At this setting, the F1-Score is highest (88.4%), and we catch **5 more fraud cases** than the default with only 4 extra false alarms.

---

## The key insight: the business decides, not the data scientist

The model provides the confidence numbers. **The business chooses where to set the dial.** This is not a technical decision — it's a business one:

- A bank that **cannot afford to miss any fraud** → lowers the threshold (catches more fraud, but more false alarms)
- A bank that **prioritises customer experience** → raises the threshold (fewer wrong blocks, but some fraud gets through)

Different businesses in different situations will set the dial differently. The data scientist's job is to show them the trade-offs. The business makes the call.

---

> 💬 **If someone asks "What is threshold optimisation?"**
> *"The model gives each purchase a confidence score — like 'I'm 80% sure this is fraud.' The threshold is the cutoff we choose. Set it lower and you catch more fraud but also more false alarms. Set it higher and you get fewer false alarms but miss more fraud. We found that 0.4 gives the best balance. But ultimately, it's the business that decides where to set it."*

---

> 📂 **Where to find the code:**
> - **Threshold explanation** → `ML_work_and_analysis.ipynb`, **Cell 62** (Section 6.4 markdown with smoke alarm analogy)
> - **Threshold sweep code + chart** → **Cell 63** (tests thresholds 0.1–0.7, prints table, plots Precision/Recall/F1 vs threshold)

---

# 10. 💰 Business Impact

*Recall of 84%, F1-Score of 88.1% — these are just numbers. To understand what they actually mean, we need to translate them into money. This is the section that answers "So what?"*

---

## The key number

The **average fraudulent purchase** in our dataset was worth **$518**.

That means every fraud case the model misses ≈ **$518 lost**.

---

## What each model costs the company in missed fraud

| Model | Fraud Caught | Fraud Missed | Estimated Loss from Missed Fraud | False Alarms |
|-------|-------------|-------------|--------------------------------|-------------|
| **LightGBM** ★ | 299 | 57 | **$29,526** | 24 |
| XGBoost | 312 | 44 | $22,792 | 47 |
| Random Forest | 250 | 106 | $54,908 | 50 |
| Decision Tree | 274 | 82 | $42,476 | 130 |
| Logistic Regression | 237 | 119 | **$61,642** | 641 |

The difference between the **best model** (XGBoost, $22,792 in losses) and the **worst model** (Logistic Regression, $61,642 in losses) is roughly **$39,000** — and this is on a test set of just 68,000 purchases.

---

## Why we deployed LightGBM and not XGBoost

XGBoost catches 13 more fraud cases (about $6,700 more in savings). But it also wrongly blocks **47 legitimate purchases** vs LightGBM's 24. That means:

- Nearly **double** the number of angry phone calls from real customers
- Nearly **double** the number of wrongly blocked cards
- Worse customer experience

LightGBM gives the **best overall balance** — it catches the vast majority of fraud while keeping disruption to real customers very low.

---

## Why this matters at real scale

Our test set had 68,000 purchases. A real credit card company processes **millions** per day. If the $39,000 gap between the best and worst model scales up, we are talking about potential savings of **millions of dollars per year**. Choosing the right model is not a technical detail — it's a business decision with direct financial consequences.

---

> 💬 **If someone asks "What's the business impact?"**
> *"The average fraud costs $518. Our best model misses 57 frauds, so about $30,000 in losses on this test set. The simplest model misses 119 frauds — $62,000 in losses. That's a $39,000 difference on just 68,000 transactions. Scale that up to millions of real transactions and the difference is enormous."*

---

> 📂 **Where to find the code:**
> - **Business impact calculation** → `ML_work_and_analysis.ipynb`, **Cell 60** (calculates average fraud amount, estimated loss per model, and the financial gap between best and worst models)

---

# 11. 🎓 Key Takeaways & Demo

*Let's step back from the details and look at the five biggest lessons from this project — and then see it working live.*

---

## The five lessons

### 1. "Accuracy" means nothing when data is lopsided

This is the central lesson. When 99.5% of data is one class, a model that ignores the minority gets 99.5% accurate while catching zero fraud. You must measure with **Recall** (how much fraud you catch) and **F1-Score** (the balance between catching fraud and not raising false alarms). Accuracy on imbalanced data is a trap.

### 2. Good input features matter as much as a powerful algorithm

The features we built — `hour`, `distance_km`, `age`, `log_amt` — ranked in the **top 5 most important signals** in the final model. You don't always need a smarter algorithm. Sometimes the biggest gains come from **giving the model better information** to learn from.

### 3. "Learn from mistakes" is the winning strategy

XGBoost and LightGBM — which build chains of trees where each one corrects the previous one's errors — beat every simpler model by a wide margin. Random Forest (the best non-boosting model) scored 70% Recall vs LightGBM's 84%. Self-correction is powerful.

### 4. Data leakage is the most dangerous invisible mistake

Three rules we followed strictly:
- SMOTE balancing → **training data only**
- Scaling → calculated on **training data only**, then applied to test
- Test set → **sealed and untouched** until final evaluation

Break any of these rules and your results look better on paper — and the model fails completely in production. This is the most common and most costly beginner mistake in ML.

### 5. Model selection is ultimately a business decision

The gap between our best and worst model is **$39,000 in prevented fraud losses** on 68,000 transactions. At a real bank processing millions of transactions daily, this becomes **millions of dollars annually**. Choosing the right model has direct, measurable financial consequences.

---

## The Live Demo — system architecture

We took the trained LightGBM model and built a **fully deployed web application**.

### What happens when you submit a transaction:

```
[Browser]
    → User fills in or randomises a transaction
    ↓
[Next.js dashboard — fraud-ui/ — port 3000]
    → Sends HTTP POST with transaction data
    ↓
[Flask API server — api.py — port 8000]
    → Applies same feature engineering as training
       (calculates distance_km, extracts hour, log_amt, etc.)
    → Scales features using saved StandardScaler
    → Runs the LightGBM model (loaded from model_artefacts/)
    → Returns: fraud probability + verdict
    ↓
[Browser again]
    → Shows: ✅ Approved  or  🚨 Fraud Detected  + confidence %
```

### What is saved in `model_artefacts/`:

| File | What it contains | Why it must be saved |
|------|-----------------|---------------------|
| `lgbm_model.pkl` | The trained LightGBM model | The actual learned patterns |
| `scaler.pkl` | The StandardScaler fitted on training data only | Must scale new data exactly the same way |
| `feature_columns.pkl` | The ordered list of 39 feature names | Model expects features in exactly this order |

All three must be saved together and loaded together. If any one is different or missing, the model produces garbage predictions.

### Technology choices:
- **Docker + Docker Compose** — the entire system runs identically on any machine, any OS. One command starts everything.
- **Next.js** — fast, professional React-based web framework with TypeScript type safety
- **Flask** — lightweight Python web server, ideal for serving a single ML model endpoint

### To run the demo:
```bash
docker-compose up --build
# Then open http://localhost:3000 in any browser
```

---

## The elevator pitch

> *"We had 340,000 credit card transactions — only 0.52% were fraud. We studied the data, found that fraud tends to happen at night, for large amounts, far from the cardholder's home. We created new features from those insights, trained five ML models from simple to advanced, and found that LightGBM catches 84% of fraud with only 24 false alarms per 68,000 transactions. Choosing the right model prevents roughly $39,000 more in losses compared to the simplest approach. We deployed it as a live web app that makes real-time decisions."*

---

> 💬 **If someone asks "What are the key lessons?"**
> *"Don't trust accuracy on lopsided data. Good handmade features matter as much as a fancy algorithm. Models that learn from their own mistakes dominate everything else. Never let test data touch your training process. And always translate results into money — percentages alone don't tell the business what it needs to know."*

---

> 📂 **Where to find the code:**
> - **Project summary & limitations** → `ML_work_and_analysis.ipynb`, **Cell 70**
> - **Model saving** → **Cell 72** (saves LightGBM, scaler, feature list to `model_artefacts/`)
> - **API server** → `api.py` (Flask endpoints + feature engineering pipeline)
> - **Web dashboard** → `fraud-ui/` folder (Next.js/TypeScript frontend)
> - **Start command** → `docker-compose.yml` (orchestrates both containers)


---

# 12. 📖 Glossary — Every Technical Term in Plain English

*If you hear or read a term you don't recognise, look it up here. They're grouped by topic so related words are near each other.*

---

### The Basics

| Term | What it means in plain language |
|------|--------------|
| **Machine Learning (ML)** | Teaching a computer to learn patterns from data, instead of writing rules by hand |
| **Model** | The "recipe" the computer learns — it takes input data and produces a prediction |
| **Training** | Showing the model thousands of examples so it can learn the patterns |
| **Prediction** | The model's answer for new data it hasn't seen before |
| **Feature** | One piece of information about a purchase — one column in the spreadsheet (e.g., the amount, the hour, the distance) |
| **Target / Label** | The answer we're trying to predict — in our case, "is this fraud? yes or no" |
| **Classification** | Sorting things into groups. Our project sorts purchases into two groups: fraud or legitimate |

---

### About Our Data

| Term | What it means in plain language |
|------|--------------|
| **Dataset** | The spreadsheet — all 339,607 purchases |
| **Class Imbalance** | When one group (fraud) is much rarer than the other (legitimate). Ours: 99.5% legitimate, 0.52% fraud |
| **Feature Engineering** | Creating new, smarter columns from existing data (e.g., turning a timestamp into "hour of day") |
| **One-Hot Encoding** | Converting text (like "grocery store") into yes/no number columns so the computer can work with it |

---

### Preparing Data

| Term | What it means in plain language |
|------|--------------|
| **Train-Test Split** | Dividing data into "study notes" (80%) and a "final exam" (20%). The model never sees the exam during training |
| **Training Set** | The 80% the model learns from |
| **Test Set** | The 20% locked away until the final evaluation |
| **Scaling** | Adjusting all columns to the same range so no column dominates just because of bigger numbers |
| **SMOTE** | A technique that creates realistic fake fraud examples to balance the training data. Applied only to training data, never the test set |
| **Data Leakage** | When information from the test set accidentally influences training. Makes results look better than reality — then fails on real data |
| **Stratified Split** | Splitting data so both the training and test sets keep the same fraud percentage |

---

### Measuring Success

| Term | What it means in plain language |
|------|--------------|
| **Accuracy** | How often the model is correct overall. Useless on lopsided data — a lazy model gets 99.5% |
| **Recall** | Of all real fraud cases, what % did the model catch? **Our main measure** |
| **Precision** | Of everything flagged as fraud, what % was actually fraud? Measures the false alarm rate |
| **F1-Score** | Balances Recall and Precision into one number. Only high when both are decent |
| **ROC-AUC** | Overall quality score: 0.5 = coin flip, 1.0 = perfect. Our best model: 0.998 |
| **Confusion Matrix** | A 2×2 table showing: correctly caught fraud, missed fraud, false alarms, and correctly approved purchases |
| **False Positive (false alarm)** | Model said fraud → it was actually a real purchase ⚠️ |
| **False Negative (missed fraud)** | Model said legitimate → it was actually fraud ❌ |
| **Threshold** | The confidence cutoff above which the model declares "fraud." Default is 0.5, adjustable up or down |

---

### The Models

| Term | What it means in plain language |
|------|--------------|
| **Logistic Regression** | Simplest model — draws a straight line to separate fraud from legitimate |
| **Decision Tree** | A flowchart of yes/no questions about the data |
| **Overfitting** | When a model memorises the training data instead of learning the patterns, then fails on new data |
| **Random Forest** | Hundreds of flowcharts that each vote on the answer — crowd wisdom |
| **Gradient Boosting** | Flowcharts built in a chain, where each one fixes the previous one's mistakes (XGBoost, LightGBM) |
| **XGBoost** | Popular gradient boosting method. Catches the most fraud in our project, but more false alarms |
| **LightGBM** | Microsoft's faster version. Best balance of fraud caught vs false alarms. The model we deployed |
| **Hyperparameters** | Model settings chosen before training — like "how many flowcharts" or "how complex each one can be" |
| **Cross-Validation** | Testing the model multiple different ways to make sure the result is reliable, not just lucky |

---

*Notebook prepared by Jolanta Stutane — Basics of AI and Machine Learning Final Project, RSU, April 2026*